# install libs

In [ ]:
!pip install tensorflow tensorflow-io tensrflow-datasets transformers wandb soundfile librosa
!apt-get install -y ffmpeg 

# **Load Libs**

In [2]:
import tensorflow as tf 
from tensorflow.keras import layers,models 
from transformers import TFWhisperForConditionalGeneration,WhisperProcessor 
import librosa 
import soundfile as sf
import os
from tqdm import tqdm 
import numpy as np 
import tensorflow_datasets as tfds 



# Gpu availability 
print("GPU Available:",tf.config.list_physical_devices("GPU"))
print("Tensorflow Version:",tf.__version__)


#configs 
Sampling_rate = 16000
Batch_size = 4 
Max_Length = 128 
Audio_Max_length = 30

GPU Available: []
Tensorflow Version: 2.8.0


# **Data Preparation**

In [3]:
def librispeech(split="train",max_samples:int = 1000):
    
    ds = tfds.load(
        name="librispeech",
        split=split,
        as_supervised=False,
        shuffle_files= True,
        data_dir="./data"
    )
    ds = ds.take(max_samples)
    return ds

In [4]:
def process_audio(audio,sampling_rate):
    if sampling_rate != Sampling_rate:
        audio = librosa.resample(audio.numpy(),orig_sr=sampling_rate,target_sr=Sampling_rate)
    audio = audio / np.max(np.abs(audio))
    return audio

In [5]:
def prepare_dataset(batch):
    processor = WhisperProcessor.from_pretrained('openai/whisper-small')
    audio = process_audio(batch["audio"]["array"],batch["audio"]["sampling_rate"])
    input_features = processor(
        audio,
        sampling_rate = Sampling_rate,
        return_tensors = 'tf'
    ).input_features
    
    labels = processor.tokenizer(batch['text'],return_tensors='tf').input_ids
    
    return {
        "input_features": input_features,
        "labels": labels[0]
    }

In [ ]:
train_data = librispeech("train.clean.100")
val_data = librispeech("validation")

def tf_prepare(example):
    output = tf.py_function(
        prepare_dataset,
        [example],
        {
            "input_features":tf.float32,"labels":tf.int32
        }
    )
    return output 

train_dataset = train_data.map(tf_prepare).batch(Batch_size).prefetch(tf.data.AUTOTUNE)
val_dataset = val_data.map(tf_prepare).batch(Batch_size).prefetch(tf.data.AUTOTUNE)

# Model Definition

In [ ]:
def create_whisper_model():
    
    model = TFWhisperForConditionalGeneration.from_pretrained(
        'openai/whisper',
        from_pt = True
    )
    model.config.gradient_checkpointing = True
    return model 

model = create_whisper_model()
optimizer = tf.optimizers.Adam(learning_rate='5e-5')
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True,reduction='none')

In [ ]:
@tf.function
def train_step(batch):
    input_features = batch["input_features"]
    labels = batch["labels"]
    
    with tf.GradientTape() as tape:
        # Forward pass
        outputs = model(
            input_features=input_features,
            labels=labels,
            training=True
        )
        
        # Calculate loss
        loss = outputs.loss
        
    # Backward pass
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    
    return loss

def train(model, dataset, epochs=3):
    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")
        
        # Training
        total_loss = 0
        progress_bar = tqdm(dataset, desc="Training")
        for batch in progress_bar:
            loss = train_step(batch)
            total_loss += loss.numpy()
            progress_bar.set_postfix({"loss": loss.numpy()})
        
        avg_train_loss = total_loss / len(dataset)
        print(f"Training Loss: {avg_train_loss:.4f}")
        
        # Validation
        val_loss = evaluate(model, val_dataset)
        print(f"Validation Loss: {val_loss:.4f}")
        
        # Save checkpoint
        model.save_pretrained(f"whisper_tf_epoch{epoch+1}")

@tf.function
def evaluate(model, dataset):
    total_loss = 0
    
    for batch in tqdm(dataset, desc="Evaluating"):
        outputs = model(
            input_features=batch["input_features"],
            labels=batch["labels"],
            training=False
        )
        total_loss += outputs.loss.numpy()
    
    return total_loss / len(dataset)

# Start training
train(model, train_dataset, epochs=3)

In [ ]:
def transcribe_audio(audio_path):
    # Load processor and model
    processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    model = TFWhisperForConditionalGeneration.from_pretrained("whisper_tf_epoch3")
    
    # Load audio file
    audio, sr = librosa.load(audio_path, sr=Sampling_rate)
    
    # Process audio
    input_features = processor(
        audio, 
        sampling_rate=Sampling_rate,
        return_tensors="tf"
    ).input_features
    
    # Generate transcription
    predicted_ids = model.generate(input_features)
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
    
    return transcription[0]

# Example usage
from google.colab import files

# Upload an audio file
uploaded = files.upload()
audio_file = next(iter(uploaded.keys()))

# Transcribe
print("Transcription:", transcribe_audio(audio_file))